In [ ]:
import mne
import mne_bids
import os, os.path as op
import nilearn
import matplotlib.pyplot as plt

from nilearn import *

In [ ]:
### This sets some environment variables that make our 3D visualizations work in this 
### particular conda environment

os.environ["QT_API"] = "pyside6"
os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"
os.environ["PYVISTA_TRAME_JUPYTER_MODE"] = "server"

# Setup Paths

## Set some generic BIDS information

In [ ]:
#Raw data
bids_root = op.join('/data/', os.environ['USER'], 'meg_workshop_data')  
#Processed Data Folder: contains (freesurfer / Day1 / Day2) 
deriv_root = op.join(bids_root, 'derivatives')
if not op.exists(deriv_root): os.mkdir(deriv_root)
#This course data - bids derivatives outputs for Day2 - hey thats today
project_root = op.join(deriv_root, 'Workshop')  
#Freesurfer brain surface reconstruction
fs_subjects_dir = op.join(deriv_root, 'freesurfer','subjects')
subjects_dir = fs_subjects_dir

### Find data for a subject

In [ ]:
subject = 'ON02811'
bids_path = mne_bids.BIDSPath(root=bids_root, subject = subject, session='01', task='airpuff', run='01', 
                        datatype='meg')
project_path = bids_path.copy().update(root=project_root, check=False)
if not op.exists(project_path.root): os.mkdir(project_path.root)

# You've seen all those previous commands before but this is new - we'll get the BIDS path for the MRI
t1_bids_path = bids_path.copy().update(datatype='anat', session='01', task=None, run=None, suffix='T1w', 
                        acquisition='MPRAGE', extension='.nii.gz')

In [ ]:
raw = mne.io.read_raw_ctf(bids_path, verbose=False, clean_names=True, system_clock='ignore')

In [ ]:
# THis is an mne_bids command that will get the transformation from the MRI to the MEG space

# It uses the anatomical landmarks that are indicated in the .json files that are in the BIDS 
# structure that are stored alongside the MRI file.

# These anatomical landmark fields are generated when you use the brainsight data and tools in 
# nih2mne

trans = mne_bids.get_head_mri_trans(bids_path, t1_bids_path=t1_bids_path, fs_subject='sub-'+subject, 
                                fs_subjects_dir=subjects_dir, 
                                extra_params=dict(clean_names=True, system_clock='ignore'))

In [ ]:
### You can ignore those warnings; mne_bids is being silly and trying to map columns from the file describing 
### the participants into MNE's subject info. 

## Transform is a 3x3 rotation matrix + the last column is the x/y/z translation

In [ ]:
trans

### Prepare Imaging for Source Localization

In [ ]:
subjects_dir

### Plot the transform

In [ ]:
project_path.directory.mkdir(parents=True, exist_ok=True)

In [ ]:
%matplotlib qt
tmp = mne.viz.plot_alignment(raw.info, trans=trans, subject='sub-'+subject, subjects_dir=subjects_dir, dig=True)

In [ ]:
### Note that the fiducial marks look really nice but the subject could have been higher in the helmet

In [ ]:
# Now we make the requird forward model and the headmodel created from the inner skull surface.
# First we'll set up the filenames

vol_fwd_fname = project_path.copy().update(extension='.fif', suffix='volfwd')
surf_fwd_fname = project_path.copy().update(extension='.fif', suffix='fwd')
bemsol_fname = vol_fwd_fname.copy().update(suffix='bem')

# Make a 1 layer bem from the inner skull - Appropriate for MEG, bad for EEG
# This takes a long time, so it has already been created for you, so we don't need to run the command
# here. You would need it, though. 
# mne.bem.make_watershed_preprocessedbem('sub-'+subject, subjects_dir=fs_subjects_dir)
    
# Next, you takes the bem_model from the inner skull surface and produces the model in the format
# MNE uses with conductivity values to each layer. 
bem = mne.make_bem_model('sub-'+subject, subjects_dir=fs_subjects_dir, conductivity=[0.3])
    
# This computes the actual solution to the forward model
bem_sol = mne.make_bem_solution(bem)
    
# Let's save the bem solution so we don't have to calculate it again
mne.bem.write_bem_solution(bemsol_fname.fpath, bem_sol, overwrite=True)
    

In [ ]:
# Let's plot the BEM
%matplotlib inline
fig=mne.viz.plot_bem(subject=f'sub-{subject}', subjects_dir=subjects_dir, brain_surfaces="white",
                orientation="coronal")
## YOu can see that the red surface (what we are concerned about) follows the contour of the brain
# and doesn't cut off chunks. So it looks good.

In [ ]:
# Now, Using the bem solution, you can now make the source space. 
# We'll make a volumetric source space first.

# note that we use the bem surface to provide a "bound" for the source space to keep it in the brain.

src_vol = mne.source_space.setup_volume_source_space(subject='sub-'+subject, subjects_dir=fs_subjects_dir,
                                                     mri='T1.mgz', bem=bem_sol) 

In [ ]:
# Now let's plot the source space

fig=mne.viz.plot_bem(src=src_vol, subject=f'sub-{subject}', subjects_dir=subjects_dir, brain_surfaces="white",
                orientation="coronal")

In [ ]:
# You can see a point for each voxel in the source space

In [ ]:
# Now let's make a surface based source space! 

src_surf = mne.source_space.setup_source_space(subject='sub-'+subject, subjects_dir=subjects_dir, n_jobs=-1) 

In [ ]:
# And plot it - we'll use a different tool

fig=mne.viz.plot_alignment(subject='sub-'+subject, subjects_dir=subjects_dir,surfaces="white",
                          coord_frame="mri", src=src_surf)

# Each yellow dot represents a source where we'll be calculating the beamformer. 
# Note that the samples aren't particularly dense. The setup_source_space function defaults to the 
# oct6 subdivision of the freesurfer surface. You can do much denser samplings, or include every
# freesurfer vertex in your solution

In [ ]:
# Now wwe can make the forward models. We'll do this separately for the volumetric source space and the 
# surface based source space

forward_surf = mne.make_forward_solution(raw.info, trans, src_surf, 
                                        bem_sol, meg=True, eeg=False, n_jobs=-1)
forward_vol = mne.make_forward_solution(raw.info, trans, src_vol, 
                                        bem_sol, meg=True, eeg=False, n_jobs=-1)

In [ ]:
surf_fwd_fname

In [ ]:
# We'll save the forward surface model, we will need that later. 
mne.forward.write_forward_solution(surf_fwd_fname, forward_surf, overwrite=True)

In [ ]:
# We can make sensitivity maps that use the lead fields to show the sensitivity
# at each point in the source space. 

In [ ]:
mag_map_surf = mne.sensitivity_map(forward_surf, ch_type="mag", mode="fixed")

In [ ]:
mag_map_vol = mne.sensitivity_map(forward_vol, ch_type="mag", mode="fixed")

In [ ]:
# For the surface model, there are differences in sensitivity based on whether you are in 
# a sulcus or gyrus

fig = mag_map_surf.plot(subjects_dir=subjects_dir, clim=dict(lims=[0, 50, 100]))

In [ ]:
# For the volumetric model, you can see that sensitivity is lower deep in the brain
# But not zero!!

fig=mag_map_vol.plot(src=src_vol, subjects_dir=subjects_dir, clim=dict(lims=[0, 10, 100]))

In [ ]:
## For the next labs, we are going to need the MRI information processed for all subjects. Let's go ahead and 
## do that so we have it ready

In [ ]:
# Okay, now lets loop over all the subjects and get the freesurfer filenames 
subjects=['ON02811','ON03748','ON22671','ON42107','ON52662','ON61373','ON62003','ON70467','ON72082',
         'ON84896','ON85305','ON89474','ON89475']
fs_subjects=[]
for subject in subjects:
    fs_subjects.append('sub-' + subject)

In [ ]:
# Now we loop over all the subjects to retrieve the trans, and make the bem, forward model, and source model.
# This might take a while. 

for subject in subjects:
    # filenames and load MRI files
    fs_subject = 'sub-' + subject
    bids_path = mne_bids.BIDSPath(root=bids_root, subject=subject, session='01',
                                  task='airpuff', run='01', datatype='meg')
    project_path = bids_path.copy().update(root=project_root, check=False)
    project_path.mkdir(exist_ok=True)
    t1_bids_path = bids_path.copy().update(
        datatype='anat', session='01', task=None, run=None,
        suffix='T1w', acquisition='MPRAGE', extension='.nii.gz')
    
    # --- output filenames (derive each from project_path) ---
    bemsol_fname = project_path.copy().update(extension='.fif', suffix='bem')
    fwd_name     = project_path.copy().update(extension='.fif', suffix='fwd')
    trans_name   = project_path.copy().update(extension='.fif', suffix='trans')
    src_name     = project_path.copy().update(extension='.fif', suffix='src')
    
    raw = mne.io.read_raw_ctf(bids_path, clean_names=True, verbose=False)
    trans = mne_bids.get_head_mri_trans(bids_path, t1_bids_path=t1_bids_path, 
                                    extra_params=dict(clean_names=True, system_clock='ignore'),
                                    fs_subject=fs_subject, fs_subjects_dir=subjects_dir)
    mne.write_trans(trans_name.fpath, trans, overwrite=True)
    
    bem = mne.make_bem_model('sub-'+subject, subjects_dir=fs_subjects_dir, conductivity=[0.3])
    bem_sol = mne.make_bem_solution(bem)
    mne.bem.write_bem_solution(bemsol_fname.fpath, bem_sol, overwrite=True)
    
    src = mne.source_space.setup_source_space(subject='sub-'+subject, subjects_dir=subjects_dir, n_jobs=4) 
    mne.source_space.write_source_spaces(src_name.fpath, src, overwrite=True)
    forward = mne.make_forward_solution(raw.info, trans, src, 
                                        bem_sol, meg=True, eeg=False, n_jobs=-1)
    mne.forward.write_forward_solution(fwd_name.fpath, forward, overwrite=True)

#### Exercise 1

Feel free to look at the bems for different subjects and see if you notice anything. It might also be interesting to look at the sensitivity maps for different subjects, and if they differ. 